# 00 — Integration

From a clean clone: config, cache, validators, counts. Runs off `data/cache/` with no network I/O once the cache exists.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))

In [2]:
import logging
import config
from src.fetch_fafb import load_neurons, load_edges, load_edgeroi
from src.validate import validate_dataset
from src.summary import summary
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
print("FAFB", config.FAFB_MATERIALIZATION, config.FAFB_ANNOTATIONS_TAG, config.FAFB_SYNAPSE_TABLE)

FAFB 783 v3.2.0 buhmann


## Load (cache first) and validate

In [3]:
neurons, edges, edgeroi = load_neurons(), load_edges(), load_edgeroi()
validate_dataset(neurons, edges, edgeroi)
summary(neurons, edges)

src.cache: cache hit: fafb_neurons_783_ann3.2.0.parquet


src.cache: cache hit: fafb_edges_783_buhmann_min1.parquet


src.cache: cache hit: fafb_edgeroi_783_buhmann_min1.parquet


,neurons,edges_min1,edges_min5,edges_min10,synapses,cell_types
dataset,,,,,,
fafb,139262,15091983,2700513,1066822,54492922,8846


## Threshold table (course grid, central brain)

In [4]:
from eda_shared.eda_helpers import add_provisional_flags, relative_weight
import pandas as pd
fl = add_provisional_flags(neurons)
central = set(fl.loc[~fl.is_optic, 'neuron_id'])
e = edges.assign(frac=relative_weight(edges))
e = e[e.pre_id.isin(central) & e.post_id.isin(central)]
rows = []
for label, keep in [(f'>= {t}', e.syn_count >= t) for t in (1, 5, 10, 31)] + \
                   [(f'>= {f:.1%} input', e.frac >= f) for f in (0.009, 0.026)]:
    k = e[keep]
    connected = pd.unique(pd.concat([k.pre_id, k.post_id]))
    rows.append({'criterion': label, 'edges': len(k), 'synapses': int(k.syn_count.sum()),
                 'isolated': len(central) - len(connected),
                 'median_out_degree': float(k.groupby('pre_id').size().median())})
pd.DataFrame(rows)

/home/pc/doktora/CENG8004_Machine_Learning/p5-pu-wiring/eda_shared/eda_helpers.py:86: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["is_optic"] = out["neuropil"].map(table["is_optic"]).fillna(False).astype(bool)


,criterion,edges,synapses,isolated,median_out_degree
0,>= 1,5886895,24883726,315,98.0
1,>= 5,1145235,17318437,1411,15.0
2,>= 10,537059,13378205,4652,6.0
3,>= 31,107536,6442081,21324,2.0
4,>= 0.9% input,1019641,11969747,353,13.0
5,>= 2.6% input,290164,5559983,1512,4.0


## Project 5 region: central complex roster

In [5]:
cx = neurons[neurons.cell_class == 'CX']
ids = set(cx.neuron_id)
ecx = edgeroi[edgeroi.neuropil.isin(['EB', 'FB', 'PB', 'NO']) & edgeroi.pre_id.isin(ids) & edgeroi.post_id.isin(ids)]
pair = ecx.groupby(['pre_id', 'post_id']).syn_count.sum()
n = len(ids)
pd.Series({'roster': n, 'ordered pairs': n * (n - 1), 'pairs >= 10': int((pair >= 10).sum()),
           'recorded below 10 (1-9 syn)': int(((pair >= 1) & (pair < 10)).sum())})

roster                            2875
ordered pairs                  8262750
pairs >= 10                      32125
recorded below 10 (1-9 syn)     257763
dtype: int64

## What this shows

Written after each run: counts match the course check (2,700,513 edges at 5 synapses), the cache works offline, and the central complex gives enough positives at the course default of 10 synapses.